In [8]:
import tensorflow as tf

class QLoRADense(tf.keras.layers.Layer):
    def __init__(self, units, rank=4):
        super().__init__()
        self.units = units
        self.rank = rank

    def build(self, input_shape):

        # we use  quantized base weight for the QLoRA
        W_float = tf.random.normal((input_shape[-1], self.units))
        self.W = tf.quantization.fake_quant_with_min_max_vars(
            W_float, min=-1, max=1
        )

        self.A = self.add_weight(
            shape=(input_shape[-1], self.rank),
            initializer="random_normal",
            trainable=True
        )

        self.B = self.add_weight(
            shape=(self.rank, self.units),
            initializer="zeros",
            trainable=True
        )

    def call(self, x):
        return tf.matmul(x, self.W) + tf.matmul(tf.matmul(x, self.A), self.B)


inputs = tf.keras.Input(shape=(128,))
x = LoRADense(64, rank=4)(inputs)
x = tf.keras.layers.ReLU()(x)
outputs = tf.keras.layers.Dense(10)(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)


x_train = np.random.randn(500,128)
y_train = np.random.randint(0,10,500)


model.fit(x_train, y_train, epochs=5, batch_size=32)

Epoch 1/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - accuracy: 0.1284 - loss: 2.3910
Epoch 2/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1124 - loss: 2.4070 
Epoch 3/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.0993 - loss: 2.3960 
Epoch 4/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1149 - loss: 2.3531 
Epoch 5/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1141 - loss: 2.3184 


In [9]:
model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lo_ra_dense_1 (LoRADense)       │ (None, 64)             │         8,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,448 (48.63 KB)

 Trainable params: 1,418 (5.54 KB)

 Non-trainable params: 8,192 (32.00 KB)

 Optimizer params: 2,838 (11.09 KB)

In [10]:
for w in model.trainable_weights:
    print(w.name, w.shape)

variable_4 (128, 4)
variable_5 (4, 64)
kernel (64, 10)
bias (10,)


In [11]:
model.count_params()

9610